<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">⚙️ Lab 01: Environment Setup & Validation</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 What We Learn

In this lab, we configure our development environment to work with **Microsoft Foundry** and the **Azure AI Projects SDK**. This is the foundation for building the **Contoso Travel** agent system — a multi-agent travel assistant that helps customers find flights, hotels, and car rentals.

---


## 🧳 The Contoso Travel Story

**Contoso Travel** is building an AI-powered travel assistant to handle the flood of customer inquiries their human advisors can no longer keep up with. The vision: a system of intelligent agents — one for flights, one for hotels, one for car rentals — that work together to plan complete trips. **You're the developer building this system**, and you've just finished setting up the Foundry project (Lab 00). Your Azure resources are provisioned, your model is deployed, and Application Insights is ready.

### 🔴 The Problem You Face Right Now

- You have a cloud project, but your **local development environment** isn't connected to it yet.
- You need to install the right SDK packages, wire up your credentials, and verify that your code can actually talk to the Foundry services.
- If this connection is broken, every lab after this one fails silently — and debugging authentication issues later, in the middle of building an agent, is painful.

### ✅ What This Lab Solves

This lab gets your development environment production-ready:
 - installing dependencies,
 - configuring environment variables,
 - validating your Azure connection,
 - testing the OpenAI client, and
 - previewing the Contoso Travel sample data you'll be working with.

By the end, running code in any subsequent lab will "just work."

## 1. Install Dependencies

We need the Azure AI Projects SDK, authentication libraries, OpenTelemetry for tracing, and pandas for working with our travel data.

---


In [ ]:
# azure-ai-projects: Foundry SDK for agents, evals, telemetry
# azure-identity: DefaultAzureCredential auth chain
# opentelemetry + azure-monitor: distributed tracing (Labs 04+)
%pip install azure-ai-projects>=2.0.0 azure-identity python-dotenv opentelemetry-sdk azure-core-tracing-opentelemetry azure-monitor-opentelemetry pandas --quiet

## 2. Configure Environment Variables

The `.env` file in `labs/notebooks/` was created during Lab 00. Verify your values match the Foundry portal:

| Variable | Where to Find |
|----------|--------------|
| `AZURE_AI_PROJECT_ENDPOINT` | Foundry portal → Project Overview |
| `AZURE_AI_MODEL_DEPLOYMENT_NAME` | Foundry portal → Models + Endpoints → Name column |
| `MODEL_ENDPOINT` | Models + endpoints → click deployment → Target URI (base URL only) |
| `MODEL_API_KEY` | Models + endpoints → click deployment → Show key |

---


In [ ]:
# Load and validate environment variables from shared .env
import os
from dotenv import load_dotenv

# abspath(".") = notebook CWD (e.g. labs/notebooks/1-prompt-agents/)
# dirname() goes up one level → labs/notebooks/ where .env lives.
# This pattern lets all lab subdirs share a single .env file.
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

required_vars = [
    "AZURE_AI_PROJECT_ENDPOINT",
    "AZURE_AI_MODEL_DEPLOYMENT_NAME",
]

# Optional vars only needed for Lab 06 (Red Teaming)
optional_vars = [
    "MODEL_ENDPOINT",
    "MODEL_API_KEY",
]

print("✅ Checking required environment variables...\n")
all_set = True
for var in required_vars:
    value = os.environ.get(var)
    if value:
        # Truncate to avoid exposing full secrets in output
        display = value[:12] + "..." + value[-4:] if len(value) > 20 else value
        print(f"  ✅ {var} = {display}")
    else:
        print(f"  ❌ {var} is NOT SET")
        all_set = False

print("\n📋 Checking optional environment variables...\n")
for var in optional_vars:
    value = os.environ.get(var)
    if value:
        display = value[:12] + "..." + value[-4:] if len(value) > 20 else value
        print(f"  ✅ {var} = {display}")
    else:
        print(f"  ⚠️  {var} is not set (needed for Red Teaming lab)")

if all_set:
    print("\n🎉 All required variables are configured!")
else:
    print("\n⚠️  Please set the missing variables in your .env file before continuing.")

## 3. Validate Azure Connection

Let's verify we can connect to your Microsoft Foundry project using the SDK.

---


In [ ]:
# Validate SDK can reach the Foundry project endpoint
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]

# Chains through az CLI, managed identity, env vars automatically
credential = DefaultAzureCredential()
# Central client for all Foundry features: agents, evals, telemetry
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

print(f"✅ Connected to Microsoft Foundry!")
print(f"   Endpoint: {endpoint}")

## 4. Validate OpenAI Client

The Azure AI Projects SDK provides an OpenAI-compatible client. Let's verify it works with a simple test prompt.

---


In [ ]:
# get_openai_client() returns a fully-configured OpenAI client
# that inherits the project endpoint and credential — no manual setup
openai_client = project_client.get_openai_client()

model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

# Uses Chat Completions API (standard across Azure OpenAI)
response = openai_client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "user", "content": "Say 'Hello from Contoso Travel!' in one sentence."}
    ]
)

print(f"✅ Model '{model_name}' is responding!")
print(f"   Response: {response.choices[0].message.content}")

## 5. Explore the Contoso Travel Sample Data

Our travel agent will use CSV data files for flights, hotels, and car rentals. Let's preview the data our agents will work with.

---


In [ ]:
# Load and preview Contoso Travel flight inventory
import pandas as pd

# Relative to notebook dir → ../../data/ reaches labs/data/
data_path = "../../data/contoso-travel"

flights = pd.read_csv(f"{data_path}/flights.csv")
print(f"✈️  Flights: {len(flights)} records")
print(f"   Routes: {flights['origin'].nunique()} origins → {flights['destination'].nunique()} destinations")
print(f"   Price range: ${flights['price_usd'].min():.0f} - ${flights['price_usd'].max():.0f}")
flights.head(3)

In [ ]:
# Preview hotel inventory across destination cities
hotels = pd.read_csv(f"{data_path}/hotels.csv")
print(f"🏨 Hotels: {len(hotels)} records")
print(f"   Cities: {', '.join(hotels['city'].unique())}")
print(f"   Star ratings: {hotels['star_rating'].min()}-{hotels['star_rating'].max()} stars")
print(f"   Price range: ${hotels['price_per_night_usd'].min():.0f} - ${hotels['price_per_night_usd'].max():.0f}/night")
hotels.head(3)

In [ ]:
# Preview car rental options by city and vehicle type
cars = pd.read_csv(f"{data_path}/car_rentals.csv")
print(f"🚗 Car Rentals: {len(cars)} records")
print(f"   Cities: {', '.join(cars['city'].unique())}")
print(f"   Types: {', '.join(cars['car_type'].unique())}")
print(f"   Price range: ${cars['price_per_day_usd'].min():.0f} - ${cars['price_per_day_usd'].max():.0f}/day")
cars.head(3)

## 6. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>Installed all required SDK dependencies</li>
  <li>Configured environment variables for Microsoft Foundry</li>
  <li>Validated connection to the Foundry project</li>
  <li>Explored the Contoso Travel sample data (flights, hotels, car rentals)</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ Next:</strong> In <a href="lab-02-agent.ipynb">Lab 02</a>, we'll create our first Prompted Agent — the Contoso Travel concierge!
</div>

---


In [ ]:
# Release HTTP connections and token caches to avoid leaks
# Order matters: close leaf clients before the parent project client
openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed. Setup complete!")